# TxnAgentBench Colab runner
Use an A100 runtime. Every experiment writes to persistent storage and can resume after disconnection.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ARTIFACT_ROOT = '/content/drive/MyDrive/txnagentbench-artifacts'


In [ ]:
import os
REPO_URL = 'https://github.com/vnigoated/programming_.git'
BRANCH = 'master'
if not os.path.exists('/content/txnagentbench/.git'):
    !git clone --branch $BRANCH $REPO_URL /content/txnagentbench
%cd /content/txnagentbench
!git fetch origin $BRANCH
!git checkout $BRANCH
!git pull --ff-only origin $BRANCH


In [ ]:
!bash scripts/bootstrap_colab.sh /content/txnagentbench
!nvidia-smi
!pytest -q


In [ ]:
SMOKE_OUT = f'{ARTIFACT_ROOT}/smoke'
!txnagentbench run --config configs/smoke.json --output $SMOKE_OUT --allow-scripted
!txnagentbench analyze --output $SMOKE_OUT
!cat $SMOKE_OUT/analysis/report.md


## Real model smoke test
Copy the pilot template, replace the model ID and immutable revision, then run only 20 executions. Do not start the primary matrix from this notebook until the runbook gates pass.

In [ ]:
PILOT_CONFIG = 'configs/pilot_local.json'  # create from pilot_template.json
MODEL_SMOKE_OUT = f'{ARTIFACT_ROOT}/model-smoke'
# !txnagentbench run --config $PILOT_CONFIG --output $MODEL_SMOKE_OUT --max-runs 20
# !txnagentbench analyze --output $MODEL_SMOKE_OUT


## Resume rule
After any Colab interruption, rerun setup and then repeat the same experiment command. Completed run IDs are skipped automatically. Never delete the artifact directory to fix a failed runtime.